# Logistic Regression Model

In [94]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

### Ordinal Encoding Categorical Data

In [121]:
df = pd.read_csv('Datasets/train.csv')

def clean(df):
    df = df.drop(['Cabin', 'Ticket', 'Name', 'PassengerId'], axis=1)
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df['Embarked'] = df['Embarked'].map({'S': 3, 'C': 2, 'Q': 1, np.nan: -1})
    df = df.fillna(-1)
    return df

df = clean(df)
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,22.0,1,0,7.2500,3
1,1,1,1,38.0,1,0,71.2833,2
2,1,3,1,26.0,0,0,7.9250,3
3,1,1,1,35.0,1,0,53.1000,3
4,0,3,0,35.0,0,0,8.0500,3


### Splitting Dataset

In [122]:
X = df.drop('Survived', axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

### Feature Scaling

In [123]:
scaler = StandardScaler()
scaler = scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

### Training Model

In [124]:
# Use Grid Search Cross Validation to find the optimal c parameter and features
rfe_lr = RFE(LogisticRegression(
    random_state = 1000,
    solver = 'lbfgs'
))
grid = GridSearchCV(
    estimator = rfe_lr,
    param_grid = {"n_features_to_select": [1, 2, 3, 4, 5, 6, 7],
                  "estimator__C": [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]
                  }
)

grid = grid.fit(X_train, y_train)
best_model = grid.best_estimator_

print(grid.best_params_)
pd.DataFrame(grid.cv_results_).head(10)

{'estimator__C': 0.01, 'n_features_to_select': 6}


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_estimator__C,param_n_features_to_select,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.008244,0.003545,0.000470,0.000177,0.0001,1,"{'estimator__C': 0.0001, 'n_features_to_select...",0.615385,0.615385,0.619718,0.612676,0.612676,0.615168,0.002578,41
1,0.004203,0.000301,0.000288,0.000016,0.0001,2,"{'estimator__C': 0.0001, 'n_features_to_select...",0.615385,0.615385,0.619718,0.612676,0.612676,0.615168,0.002578,41
2,0.003235,0.000195,0.000285,0.000053,0.0001,3,"{'estimator__C': 0.0001, 'n_features_to_select...",0.615385,0.615385,0.619718,0.612676,0.612676,0.615168,0.002578,41
3,0.002544,0.000081,0.000230,0.000006,0.0001,4,"{'estimator__C': 0.0001, 'n_features_to_select...",0.615385,0.615385,0.619718,0.612676,0.612676,0.615168,0.002578,41
4,0.001937,0.000036,0.000233,0.000008,0.0001,5,"{'estimator__C': 0.0001, 'n_features_to_select...",0.615385,0.615385,0.619718,0.612676,0.612676,0.615168,0.002578,41
5,0.001350,0.000028,0.000232,0.000009,0.0001,6,"{'estimator__C': 0.0001, 'n_features_to_select...",0.615385,0.615385,0.619718,0.612676,0.612676,0.615168,0.002578,41
6,0.000714,0.000025,0.000225,0.000009,0.0001,7,"{'estimator__C': 0.0001, 'n_features_to_select...",0.615385,0.615385,0.619718,0.612676,0.612676,0.615168,0.002578,41
7,0.003604,0.000142,0.000226,0.000004,0.0010,1,"{'estimator__C': 0.001, 'n_features_to_select'...",0.615385,0.615385,0.619718,0.612676,0.612676,0.615168,0.002578,41
8,0.003222,0.000048,0.000229,0.000003,0.0010,2,"{'estimator__C': 0.001, 'n_features_to_select'...",0.615385,0.615385,0.619718,0.612676,0.612676,0.615168,0.002578,41
9,0.002724,0.000086,0.000228,0.000003,0.0010,3,"{'estimator__C': 0.001, 'n_features_to_select'...",0.629371,0.629371,0.626761,0.640845,0.626761,0.630621,0.005243,40


In [125]:
# Use the best model to predict the test data

df_test = pd.read_csv("Datasets/test.csv")
passenger = df_test['PassengerId']
df_test = clean(df_test)
df_test = scaler.transform(df_test)
prediction = best_model.predict(df_test)

results = pd.DataFrame({"PassengerId": passenger, "Survived": prediction})
results.head(10)

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0


In [127]:
results.to_csv('Datasets/LogisticRegression.csv', index = False)